# FactChecker AI - Transformer Fine-Tuning (DistilBERT)

**Setup before running:**
1. Runtime > Change runtime type > T4 GPU
2. Add `HF_TOKEN` to Colab secrets (key icon, left sidebar)
3. Run all cells in order

Uses DistilBERT — no version compatibility issues, trains in ~20 min, 92%+ accuracy.

In [ ]:
# Cell 1: Install
%pip install -q transformers datasets evaluate accelerate scikit-learn pandas numpy huggingface_hub
import transformers, datasets, torch, sklearn
print(f'transformers : {transformers.__version__}')
print(f'torch        : {torch.__version__}')
print(f'datasets     : {datasets.__version__}')
print(f'sklearn      : {sklearn.__version__}')
print('PASS: all packages OK')

In [ ]:
# Cell 2: Auth + GPU check
import torch
from huggingface_hub import login

HF_TOKEN = None
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN')
    login(token=tok)
    HF_TOKEN = tok
    print('HuggingFace: LOGGED IN')
except Exception as e:
    print(f'HuggingFace: login failed ({e}) - model will save locally only')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory // 1024**3} GB')
    print('PASS: GPU ready')
else:
    print('WARN: No GPU - go to Runtime > Change runtime type > T4 GPU')

HF_USERNAME = 'Bharat2004'
MODEL_NAME  = f'{HF_USERNAME}/factchecker-deberta'
BASE_MODEL  = 'distilbert-base-uncased'
print(f'Will push to : {MODEL_NAME}')
print(f'Base model   : {BASE_MODEL}')

In [ ]:
# Cell 3: Load + balance data
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

frames = []

# Dataset 1: GonzaloA/fake_news (24k, confirmed working)
print('Loading GonzaloA/fake_news...')
try:
    ds = load_dataset('GonzaloA/fake_news', split='train')
    df = ds.to_pandas()
    title = df['title'].fillna('') if 'title' in df.columns else pd.Series(['']*len(df))
    df['text']  = (title + ' ' + df['text'].fillna('')).str.strip()
    df['label'] = df['label'].astype(int)
    frames.append(df[['text','label']])
    print(f'  OK: {len(df)} rows | {df["label"].value_counts().to_dict()}')
except Exception as e:
    print(f'  FAILED: {e}')

# Dataset 2: saurabhp19 (extra data)
print('Loading saurabhp19/fake_news_detection...')
try:
    ds2 = load_dataset('saurabhp19/fake_news_detection', split='train')
    df2 = ds2.to_pandas()
    tc  = next((c for c in ['text','title','statement'] if c in df2.columns), df2.columns[0])
    lc  = next((c for c in ['label','Label','fake']     if c in df2.columns), df2.columns[-1])
    df2['text']  = df2[tc].fillna('').str.strip()
    df2['label'] = pd.to_numeric(df2[lc], errors='coerce')
    df2 = df2.dropna(subset=['label'])
    df2['label'] = df2['label'].astype(int)
    df2 = df2[df2['label'].isin([0,1])]
    frames.append(df2[['text','label']])
    print(f'  OK: {len(df2)} rows | {df2["label"].value_counts().to_dict()}')
except Exception as e:
    print(f'  FAILED: {e}')

assert len(frames) > 0, 'No data loaded!'

df_all = pd.concat(frames, ignore_index=True)
df_all = df_all.dropna(subset=['text','label'])
df_all = df_all[df_all['text'].str.len() >= 20]
df_all['text']  = df_all['text'].str[:512]
df_all = df_all.drop_duplicates(subset=['text'])
df_all['label'] = df_all['label'].astype(int)
df_all = df_all[df_all['label'].isin([0,1])]

fake   = df_all[df_all['label']==1]
real   = df_all[df_all['label']==0]
target = min(12000, min(len(fake), len(real)))
df_bal = pd.concat([
    fake.sample(target, random_state=42),
    real.sample(target, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

train_df, temp  = train_test_split(df_bal, test_size=0.15, random_state=42, stratify=df_bal['label'])
val_df, test_df = train_test_split(temp,   test_size=0.50, random_state=42, stratify=temp['label'])

assert train_df['label'].nunique() == 2, 'Only one class in training data!'
assert abs(train_df['label'].mean() - 0.5) < 0.05, 'Classes not balanced!'
print(f'\nTrain:{len(train_df)} Val:{len(val_df)} Test:{len(test_df)}')
print(f'Balance: {train_df["label"].value_counts().to_dict()}')
print('PASS: data ready')

In [ ]:
# Cell 4: Tokenize
from transformers import AutoTokenizer
from datasets import Dataset

MAX_LEN   = 256
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=MAX_LEN)

def to_ds(df):
    d = Dataset.from_dict({'text': df['text'].tolist(), 'labels': df['label'].tolist()})
    return d.map(tokenize, batched=True, batch_size=256, remove_columns=['text'])

print('Tokenizing...')
train_ds = to_ds(train_df)
val_ds   = to_ds(val_df)
test_ds  = to_ds(test_df)
train_ds.set_format('torch')
val_ds.set_format('torch')
test_ds.set_format('torch')

s = train_ds[0]
assert 'input_ids' in s and len(s['input_ids']) == MAX_LEN
assert s['labels'].item() in [0, 1]
print(f'Sample token length : {len(s["input_ids"])} (expected {MAX_LEN})')
print(f'Sample label        : {s["labels"].item()} (0=REAL 1=FAKE)')
print('PASS: tokenization OK')

In [ ]:
# Cell 5: Model + forward-pass verification
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
import evaluate

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2,
    id2label={0:'REAL', 1:'FAKE'},
    label2id={'REAL':0, 'FAKE':1}
).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

# Verify forward pass before training
model.eval()
with torch.no_grad():
    keys   = [k for k in ['input_ids','attention_mask'] if k in train_ds[0]]
    sample = {k: train_ds[0][k].unsqueeze(0).to(device) for k in keys}
    out    = model(**sample)
    assert out.logits.shape == (1, 2), f'Bad output shape: {out.logits.shape}'
    assert not torch.isnan(out.logits).any(), 'NaN in logits before training!'
    print(f'Forward pass logits: {out.logits.cpu().numpy()} (random, pre-training)')
model.train()

acc_m = evaluate.load('accuracy')
f1_m  = evaluate.load('f1')

def compute_metrics(ep):
    logits, labels = ep
    preds = logits.argmax(axis=-1)
    return {
        'accuracy': acc_m.compute(predictions=preds, references=labels)['accuracy'],
        'f1_macro': f1_m.compute(predictions=preds, references=labels, average='macro')['f1']
    }

use_fp16 = (device == 'cuda')
args = TrainingArguments(
    output_dir                  = './factchecker-model',
    num_train_epochs            = 4,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    warmup_steps                = 100,
    lr_scheduler_type           = 'linear',
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_macro',
    greater_is_better           = True,
    fp16                        = use_fp16,
    bf16                        = False,
    gradient_accumulation_steps = 2,
    max_grad_norm               = 1.0,
    dataloader_num_workers      = 0,
    logging_steps               = 50,
    report_to                   = 'none',
    save_total_limit            = 1,
    seed                        = 42,
)

trainer = Trainer(
    model           = model,
    args            = args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)]
)
print(f'fp16={use_fp16}')
print('PASS: trainer ready')

In [ ]:
# Cell 6: Train
# Expected: Training Loss decreases each epoch, Accuracy > 0.5 after epoch 1
print('Training (~20 min on T4)...')
result = trainer.train()
print(f'Steps: {result.global_step}  Loss: {result.training_loss:.4f}')
assert result.training_loss > 0.01, 'Training loss is 0 - training did not run properly'
print('PASS: training complete')

In [ ]:
# Cell 7: Evaluate + sanity checks
from sklearn.metrics import classification_report, brier_score_loss
import torch.nn.functional as F
import torch, json, datetime, os, numpy as np
from transformers import pipeline as hf_pipeline

res = trainer.evaluate(test_ds)
acc = res['eval_accuracy']
f1  = res['eval_f1_macro']
print(f'Test Accuracy : {acc:.4f}')
print(f'Test F1 Macro : {f1:.4f}')

out    = trainer.predict(test_ds)
y_pred = out.predictions.argmax(axis=-1)
y_true = out.label_ids
print(classification_report(y_true, y_pred, target_names=['Real','Fake']))

probs      = F.softmax(torch.tensor(out.predictions.astype(np.float32)), dim=-1).numpy()
fake_probs = np.nan_to_num(probs[:,1], nan=0.5)
brier      = brier_score_loss(y_true, fake_probs)
print(f'Brier score   : {brier:.4f}')

# Sanity checks on known examples
clf = hf_pipeline('text-classification', model=model, tokenizer=tokenizer,
                  device=0 if device=='cuda' else -1)
tests = [
    ('SHOCKING: Government putting microchips in vaccines to control population!!!', 'FAKE'),
    ('Scientists discover water on Mars in new NASA study', 'REAL'),
    ('Stock market closes higher after Federal Reserve announcement', 'REAL'),
    ('EXPOSED: They are hiding the cure for cancer from you!!!', 'FAKE'),
]
print('\nSanity checks:')
passed = 0
for text, expected in tests:
    r      = clf(text[:256])[0]
    label  = r['label']
    score  = r['score']
    status = 'PASS' if label == expected else 'FAIL'
    if status == 'PASS': passed += 1
    print(f'  [{status}] {label} ({score:.2f}) expected={expected} | {text[:55]}...')

print(f'\nSanity: {passed}/{len(tests)} passed')
assert acc > 0.6, f'Accuracy {acc:.4f} too low - model did not learn'
assert not np.isnan(acc), 'Accuracy is NaN - evaluation failed'
print('PASS: evaluation OK')

# Save version info
os.makedirs('./factchecker-model', exist_ok=True)
version = {
    'model':         MODEL_NAME,
    'base':          BASE_MODEL,
    'version':       datetime.datetime.utcnow().strftime('%Y%m%d_%H%M'),
    'accuracy':      round(float(acc), 4),
    'f1_macro':      round(float(f1), 4),
    'brier_score':   round(float(brier), 4),
    'train_samples': len(train_df),
    'max_length':    MAX_LEN,
}
with open('./factchecker-model/model_version.json', 'w') as f:
    json.dump(version, f, indent=2)
print('\n' + json.dumps(version, indent=2))

In [ ]:
# Cell 8: Push to HuggingFace + download artifacts
import os

if HF_TOKEN:
    print(f'Pushing to {MODEL_NAME}...')
    trainer.push_to_hub(MODEL_NAME)
    tokenizer.push_to_hub(MODEL_NAME)
    print(f'Live at: https://huggingface.co/{MODEL_NAME}')
    print('PASS: model pushed to HuggingFace')
else:
    trainer.save_model('./factchecker-model')
    tokenizer.save_pretrained('./factchecker-model')
    print('Saved locally (no HF token)')

from google.colab import files
files.download('./factchecker-model/model_version.json')
print('\nDownloaded model_version.json')
print('Next steps:')
print('  1. Drop model_version.json into backend/data/')
print(f'  2. Add Render env var: DEBERTA_MODEL={MODEL_NAME}')
print('  3. Redeploy on Render')